# Gold 04 – Driver Lap Analytics


**Source:** `formula1_dev.silver.lap_times`

Actual columns used: `race_id`, `driver_id`, `lap`, `milliseconds_clean`

**Output:** `formula1_dev.gold.driver_lap_analytics`

Scenarios: previous lap, next lap, running total, remaining total, driver total, driver average, first value, last value.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "formula1_dev"

lap_times_df = spark.table(f"{catalog}.silver.lap_times")


In [0]:
lap_window = (
    Window
    .partitionBy("driver_id")
    .orderBy("race_id", "lap")
)

running_window = (
    Window
    .partitionBy("driver_id")
    .orderBy("race_id", "lap")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

remaining_window = (
    Window
    .partitionBy("driver_id")
    .orderBy("race_id", "lap")
    .rowsBetween(Window.currentRow, Window.unboundedFollowing)
)

full_window = (
    Window
    .partitionBy("driver_id")
    .orderBy("race_id", "lap")
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
)

driver_window = Window.partitionBy("driver_id")


In [0]:
driver_lap_analytics = (
    lap_times_df
    .withColumn("previous_lap_time",
                F.lag("milliseconds_clean").over(lap_window))
    .withColumn("next_lap_time",
                F.lead("milliseconds_clean").over(lap_window))
    .withColumn("lap_time_difference",
                F.col("milliseconds_clean") - F.col("previous_lap_time"))
    .withColumn("running_lap_time",
                F.sum("milliseconds_clean").over(running_window))
    .withColumn("remaining_lap_time",
                F.sum("milliseconds_clean").over(remaining_window))
    .withColumn("driver_total_lap_time",
                F.sum("milliseconds_clean").over(driver_window))
    .withColumn("driver_avg_lap_time",
                F.avg("milliseconds_clean").over(driver_window))
    .withColumn("first_lap_time",
                F.first_value("milliseconds_clean").over(full_window))
    .withColumn("last_lap_time",
                F.last_value("milliseconds_clean").over(full_window))
    .withColumn("gold_processed_timestamp", F.current_timestamp())
)

driver_lap_analytics.display()


In [0]:
(
    driver_lap_analytics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.driver_lap_analytics")
)
